In [1]:
from huggingface_hub import hf_hub_download

repo_id = "Sarthak2910/transformer-en-hi"

model_path = hf_hub_download(repo_id=repo_id, filename="pytorch_model.pt")
tokenizer_en_path = hf_hub_download(repo_id=repo_id, filename="tokenizer_en.json")
tokenizer_hi_path = hf_hub_download(repo_id=repo_id, filename="tokenizer_hi.json")

/home/sarthak/miniconda3/envs/torch310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import torch
from tokenizers import Tokenizer
from config import get_config
from train import get_model
from train import greedy_decode

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = get_config()

tokenizer_src = Tokenizer.from_file(tokenizer_en_path)
tokenizer_tgt = Tokenizer.from_file(tokenizer_hi_path)

model = get_model(
    config,
    tokenizer_src.get_vocab_size(),
    tokenizer_tgt.get_vocab_size()
).to(device)

state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)

model.eval()

Transformer(
  (encoder): Encoder(
    (layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (self_attention_block): MultiHeadAttentionBlock(
          (w_q): Linear(in_features=512, out_features=512, bias=False)
          (w_k): Linear(in_features=512, out_features=512, bias=False)
          (w_v): Linear(in_features=512, out_features=512, bias=False)
          (w_o): Linear(in_features=512, out_features=512, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (feed_forward_block): FeedForwardBlock(
          (linear_1): Linear(in_features=512, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear_2): Linear(in_features=2048, out_features=512, bias=True)
        )
        (residual_connections): ModuleList(
          (0-1): 2 x ResidualConnection(
            (dropout): Dropout(p=0.1, inplace=False)
            (norm): LayerNormalization()
          )
        )
      )
    )
    (norm): LayerNormalization

In [7]:
def translate_sentence(sentence, model, tokenizer_src, tokenizer_tgt, config, device):
    model.eval()

    src_tokens = tokenizer_src.encode(sentence).ids

    sos_id = tokenizer_src.token_to_id('[SOS]')
    eos_id = tokenizer_src.token_to_id('[EOS]')
    pad_id = tokenizer_src.token_to_id('[PAD]')

    src_tokens = [sos_id] + src_tokens + [eos_id]

    if len(src_tokens) < config['seq_len']:
        src_tokens += [pad_id] * (config['seq_len'] - len(src_tokens))
    else:
        src_tokens = src_tokens[:config['seq_len']]

    source = torch.tensor(src_tokens).unsqueeze(0).to(device)

    source_mask = (source != pad_id).unsqueeze(1).unsqueeze(2).int().to(device)

    output_tokens = greedy_decode(
        model, source, source_mask,
        tokenizer_src, tokenizer_tgt,
        config['seq_len'], device
    )
    translated = tokenizer_tgt.decode(output_tokens.detach().cpu().numpy())

    return translated

In [8]:
output = translate_sentence(
    "How are you?",
    model,
    tokenizer_src,
    tokenizer_tgt,
    config,
    device
)

print(output)

आप किस तरह ?
